In [1]:
#!pip install dcor

In [2]:
# ================================================================
# DETECTOR DE ASSOCIAÇÕES
# ================================================================
# O QUE ESTE ARQUIVO FAZ?
# - Percorre todos os pares de colunas de um DataFrame (df)
# - Escolhe automaticamente a medida de associação apropriada
#   para cada tipo de par (num×num, num×bin, num×cat/ord, cat×cat)
# - Calcula efeito, p-valor (quando aplicável) e corrige múltiplos
#   testes com FDR (Benjamini–Hochberg)
# - Opcionalmente executa testes por permutação (mais robustos)
# - Testa não-linearidade entre variáveis numéricas (spline vs linear)
# - Retorna uma tabela ordenada por significância (q-value)
#
# POR QUE EXISTE?
# - “Varredura” inicial (“data screening”) para encontrar relações
#   interessantes, evitando *p-hacking* (usa FDR) e cobrindo casos
#   comuns de dados tabulares (numéricos e categóricos).
#
# ATENÇÃO
# - Associação ≠ causalidade.
# - p-valor depende de pressupostos; a permutação relaxa alguns deles.
# - FDR controla falsos positivos “em média”
# ================================================================

import warnings
import numpy as np
import pandas as pd
from scipy import stats

# Funcoes

In [3]:
# ------------------------------------------------
# [BLOCO 1] Dependências opcionais
# ------------------------------------------------
# - Se bibliotecas opcionais estiverem presentes, usamos (resultados melhores).
# - Se não estiverem, o código funciona no "modo reduzido" (sem quebrar).
try:
    import dcor                          # Distance correlation (capta relações não-lineares gerais)
    _HAS_DCOR = True
except Exception:
    _HAS_DCOR = False
    warnings.warn("Pacote 'dcor' não encontrado: distance correlation ficará indisponível.")

try:
    import statsmodels.api as sm         # Modelagem (OLS), ANOVA, covariâncias robustas
    import statsmodels.formula.api as smf
    _HAS_SM = True
except Exception:
    _HAS_SM = False
    warnings.warn("Pacote 'statsmodels' não encontrado: teste spline/linear ficará indisponível.")

try:
    from patsy import dmatrix            # Base spline (B-spline) via fórmulas
    _HAS_PATSY = True
except Exception:
    _HAS_PATSY = False
    warnings.warn("Pacote 'patsy' não encontrado: geração de splines indisponível.")

In [4]:
# ----------------------------------------------------------
# [BLOCO 2] Funções utilitárias (reuso em toda a pipeline)
# ----------------------------------------------------------
def _kind(s: pd.Series) -> str:
    """
    Classifica uma variável em: numeric, binary, categorical, ordinal.
    Conceito:
      - Binária: apenas 2 valores distintos.
      - Ordinal: ex. escalas inteiras de 0 a 5; também CategoricalDtype(ordered=True).
      - Categórica: objetos/strings/categorias sem ordem.
      - Numérica: float/inteiro com muitas categorias (contínua).
    """
    s_no_na = s.dropna()
    u = s_no_na.nunique()

    if u == 2:
        return "binary"

    # Categorical(ordered=True) → ordinal
    if pd.api.types.is_categorical_dtype(s) and getattr(s.dtype, "ordered", False):
        return "ordinal"

    # Numérica com poucos valores inteiros → ordinal (útil p/ escalas Likert)
    if pd.api.types.is_numeric_dtype(s):
        if np.allclose(s_no_na, np.round(s_no_na)) and u <= 7:
            return "ordinal"
        return "numeric"

    # Objetos/strings/categorias → categórica
    if pd.api.types.is_categorical_dtype(s) or pd.api.types.is_object_dtype(s):
        return "categorical"

    # Padrão conservador
    return "categorical"


def _align_no_na(x: pd.Series, y: pd.Series):
    """
    Alinha x e y e remove linhas com NA nos dois ao mesmo tempo.
    Por quê? A maioria dos testes assume pares (xi, yi) completos.
    """
    df = pd.concat([x, y], axis=1).dropna()
    return df.iloc[:, 0], df.iloc[:, 1]


def _bh_correction(pvals):
    """
    Benjamini–Hochberg (FDR)
    Conceito: controla a taxa de falsos descobrimentos ao fazer muitos testes.
    Retorna q-values (p ajustados). Implementação manual para evitar dependências.
    """
    p = np.asarray(list(pvals), dtype=float)
    if p.size == 0:
        return p
    m = len(p)                      # número de testes
    order = np.argsort(p)
    ranked = p[order]
    # BH: q_i = p_(i) * m / i, após impor monotonicidade
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]  # força q_i não-decrescente ao longo das ordens
    out = np.empty_like(q)
    out[order] = np.clip(q, 0, 1)
    return out

In [5]:
# --------------------------------------------------------------------
# [BLOCO 3] Medidas por tipo de par (funções pequenas e focadas)
# --------------------------------------------------------------------
def _num_num(x, y, *, permutations=0, spline_test="wald_robust", df_spline=5, degree=3):
    """
    Par NUMÉRICO × NUMÉRICO.
    Retorna lista de resultados com:
      - Pearson (associação linear, sensível a outliers)
      - Spearman (correlação de postos: monotônica)
      - Kendall tau (acordos de ordem: monotônica, robusta)
      - Distance correlation (dCor) [se disponível] (capta dependência geral)
      - Spline vs Linear [se disponível] (testa não-linearidade)
        > 'anova': ANOVA entre modelos (linear vs spline)
        > 'wald_robust': Wald conjunto dos coeficientes spline com HC3 (robusto à heteroscedasticidade)
    """
    out = []

    # 3.1 Pearson
    r, p = stats.pearsonr(x, y)
    out.append(dict(method="pearson", effect=float(r), p_value=float(p)))

    # 3.2 Spearman (ordens)
    rs, ps = stats.spearmanr(x, y, nan_policy="omit")
    out.append(dict(method="spearman", effect=float(rs), p_value=float(ps)))

    # 3.3 Kendall tau
    tau, pk = stats.kendalltau(x, y, nan_policy="omit")
    out.append(dict(method="kendall_tau", effect=float(tau), p_value=float(pk)))

    # 3.4 Distance correlation (opcional): capta dependências não-lineares e não-monotônicas
    if _HAS_DCOR:
        dc = float(dcor.distance_correlation(x.values, y.values))
        if permutations and permutations > 0:
            # p-valor por permutação: embaralha y para simular a hipótese nula de independência
            rng = np.random.default_rng(123)
            more = 0
            for _ in range(int(permutations)):
                dc_perm = float(dcor.distance_correlation(x.values, rng.permutation(y.values)))
                if dc_perm >= dc:
                    more += 1
            pdc = (more + 1) / (permutations + 1)
        else:
            pdc = np.nan
        out.append(dict(method="distance_corr", effect=dc, p_value=float(pdc)))
    else:
        out.append(dict(method="distance_corr", effect=np.nan, p_value=1.0, notes="dcor ausente"))

    # 3.5 Spline vs Linear (opcional): detecta curvaturas (não-linearidade)
    if _HAS_SM and _HAS_PATSY:
        df_xy = pd.DataFrame({"y": y.values, "x": x.values}).dropna()

        # Modelo linear simples: y ~ x
        m_lin = smf.ols("y ~ x", data=df_xy).fit()

        # Base spline (B-splines) para permitir curvatura: y ~ x + spline(x)
        bs = dmatrix(f"bs(x, df={df_spline}, degree={degree}, include_intercept=False)",
                     df_xy, return_type="dataframe")
        bs.columns = [f"spline_{i}" for i in range(bs.shape[1])]
        df2 = pd.concat([df_xy.reset_index(drop=True), bs.reset_index(drop=True)], axis=1)
        form = "y ~ x + " + " + ".join(bs.columns)
        m_spl = smf.ols(form, data=df2).fit()
        r2 = float(m_spl.rsquared)  # efeito (aqui uso o R² do modelo mais flexível)

        # Teste estatístico: spline melhora em relação ao linear?
        if spline_test == "anova":
            # ANOVA de modelos aninhados
            _, p_spl, _ = m_spl.compare_f_test(m_lin)
            out.append(dict(method="spline_vs_linear_anova", effect=r2, p_value=float(p_spl)))
        else:
            # Wald conjunto (HC3) para os termos spline != 0 (robusto a heteroscedasticidade)
            m_rob = smf.ols(form, data=df2).fit(cov_type="HC3")
            params = m_rob.params.index.tolist()
            idx = [params.index(c) for c in bs.columns if c in params]
            if len(idx) == 0:
                out.append(dict(method="spline_vs_linear_waldHC3", effect=r2, p_value=np.nan, notes="sem termos spline"))
            else:
                R = np.zeros((len(idx), len(params)))
                for j, k in enumerate(idx):
                    R[j, k] = 1.0
                wald = m_rob.wald_test(R)
                p_spl = float(np.asarray(wald.pvalue).ravel()[0])
                out.append(dict(method="spline_vs_linear_waldHC3", effect=r2, p_value=p_spl))
    else:
        out.append(dict(method="spline_vs_linear", effect=np.nan, p_value=1.0, notes="statsmodels/patsy ausentes"))

    return out


def _num_bin(xn, yb):
    """
    Par NUMÉRICO × BINÁRIO.
    Conceito: correlação point-biserial (equivalente à correlação de Pearson
    entre uma variável numérica e uma variável 0/1).
    """
    # Se a série binária não for numérica, converte para códigos 0/1
    if not pd.api.types.is_numeric_dtype(yb):
        yb = yb.astype("category").cat.codes
    # SciPy oferece implementação direta; em caso raro de ausência, Pearson é equivalente
    try:
        r, p = stats.pointbiserialr(xn, yb)
    except Exception:
        r, p = stats.pearsonr(xn, yb)
    return [dict(method="point_biserial", effect=float(r), p_value=float(p))]


def _cat_cat(xc, yc):
    """
    Par CATEGÓRICO × CATEGÓRICO (inclui bin×bin e ord×cat).
    Conceito: Cramér's V (0 a 1), derivado do qui-quadrado de independência.
              Aqui usamos CORREÇÃO DE VIÉS (recomendada p/ tabelas pequenas/esparsas).
    """
    tbl = pd.crosstab(xc, yc)
    chi2, p, _, _ = stats.chi2_contingency(tbl, correction=False)
    n = tbl.to_numpy().sum()
    r, k = tbl.shape

    # Correção de viés (Bergsma & Wicher / ajuste para tabelas finitas)
    phi2 = max(0.0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    k_corr = k - (k - 1) ** 2 / (n - 1)
    r_corr = r - (r - 1) ** 2 / (n - 1)
    denom = max(1.0, min(k_corr - 1, r_corr - 1))
    v = np.sqrt(phi2 / denom) if denom > 0 else 0.0

    return [dict(method="cramers_v", effect=float(v), p_value=float(p))]


def _num_cat(num, cat, *, permutations=0):
    """
    Par NUMÉRICO × CATEGÓRICO/ORDINAL.
    Conceito: Eta² (η²), tamanho de efeito da ANOVA (proporção da variância explicada por grupos).
              Usamos p-valor por permutação (embaralhando grupos) para reduzir suposições.
    """
    df = pd.DataFrame({"y": num, "g": cat}).dropna()
    y = df["y"].values
    groups = [grp.values for _, grp in df.groupby("g")["y"]]

    # Soma de quadrados entre grupos (SSB) e total (SST)
    grand = np.mean(y)
    ssb = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)
    sst = np.sum((y - grand) ** 2)
    eta2 = float(ssb / sst) if sst > 0 else 0.0

    # p-valor via permutação (opcional)
    if permutations and permutations > 0:
        rng = np.random.default_rng(123)
        more = 0
        for _ in range(int(permutations)):
            g_perm = rng.permutation(df["g"].values)
            groups_p = [df.loc[g_perm == v, "y"].values for v in np.unique(g_perm)]
            ssb_p = sum(len(g) * (g.mean() - grand) ** 2 for g in groups_p)
            eta2_p = float(ssb_p / sst) if sst > 0 else 0.0
            if eta2_p >= eta2:
                more += 1
        p = (more + 1) / (permutations + 1)
    else:
        p = np.nan

    return [dict(method="eta_squared", effect=eta2, p_value=float(p))]

In [6]:
# -----------------------------------------------------------------
# [BLOCO 4] Função principal de varredura
# -----------------------------------------------------------------
def scan_associations(
    df: pd.DataFrame,
    *,
    permutations: int = 500,          # quantas permutações (0 = desliga)
    fdr_mode: str = "by_method",    # "global" | "by_method" (recomendado) | "by_pair"
    spline_test: str = "wald_robust",  # "anova" | "wald_robust"
    df_spline: int = 5,
    degree: int = 3,
    progress: bool = True
) -> pd.DataFrame:
    """
    Varre todos os pares de colunas de df e retorna um DataFrame com:
      x, y, type_pair, method, effect, p_value, q_value, significant_q<0.05, n, notes

    Lógica:
      1) Detecta o tipo de cada variável (numeric, binary, categorical, ordinal)
      2) Para cada par (x, y), escolhe o bloco de testes adequado
      3) Junta todos os resultados em uma tabela única
      4) Aplica correção FDR conforme o modo escolhido
      5) Ordena pela significância (q-value/p-value)
    """
    # Iteração com barra de progresso (se a lib tqdm existir)
    try:
        from tqdm.auto import tqdm
        pairs_iter = tqdm([(a, b) for i, a in enumerate(df.columns) for b in df.columns[i + 1:]],
                          disable=not progress, desc="Pares")
    except Exception:
        pairs_iter = [(a, b) for i, a in enumerate(df.columns) for b in df.columns[i + 1:]]

    kinds = {c: _kind(df[c]) for c in df.columns}   # cache do tipo
    rows = []

    for a, b in pairs_iter:
        # 1) alinha e remove NAs
        xa, yb = _align_no_na(df[a], df[b])
        if len(xa) < 3:   # casos minúsculos não são informativos
            continue

        # 2) tipos do par
        ka, kb = kinds[a], kinds[b]
        n = len(xa)

        # 3) roteamento simples por tipo de par
        try:
            if ka == "numeric" and kb == "numeric":
                results = _num_num(xa, yb, permutations=permutations,
                                   spline_test=spline_test, df_spline=df_spline, degree=degree)
                tpair = "numeric-numeric"

            elif (ka == "numeric" and kb == "binary") or (ka == "binary" and kb == "numeric"):
                xn = xa if ka == "numeric" else yb
                ybin = yb if kb == "binary" else xa
                results = _num_bin(xn, ybin)
                tpair = "numeric-binary"

            elif (ka in ("categorical", "ordinal") and kb in ("categorical", "ordinal")):
                results = _cat_cat(xa.astype("category"), yb.astype("category"))
                tpair = "categorical-categorical"

            elif (ka == "numeric" and kb in ("categorical", "ordinal")) or (kb == "numeric" and ka in ("categorical", "ordinal")):
                num = xa if ka == "numeric" else yb
                cat = yb if kb in ("categorical", "ordinal") else xa
                results = _num_cat(num, cat.astype("category"), permutations=permutations)
                tpair = "numeric-categorical"

            elif ka == "binary" and kb == "binary":
                results = _cat_cat(xa.astype("category"), yb.astype("category"))
                tpair = "binary-binary"

            else:
                # fallback seguro: trata como categórico×categórico (Cramér V)
                results = _cat_cat(xa.astype("category"), yb.astype("category"))
                tpair = f"{ka}-{kb}"

        except Exception as e:
            # Em caso de erro pontual em um par, registramos e seguimos
            results = [dict(method="error", effect=np.nan, p_value=np.nan, notes=str(e))]
            tpair = f"{ka}-{kb}"

        # 4) agrega resultados deste par
        for r in results:
            rows.append(dict(
                x=a, y=b, type_pair=tpair, method=r.get("method", ""),
                effect=r.get("effect", np.nan), p_value=r.get("p_value", np.nan),
                n=n, notes=r.get("notes", "")
            ))

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    # 5) correção FDR (Benjamini–Hochberg)
    def _apply_bh(df_in: pd.DataFrame) -> pd.DataFrame:
        mask = df_in["p_value"].notna()
        q = np.full(len(df_in), np.nan)
        q[mask] = _bh_correction(df_in.loc[mask, "p_value"].values)
        df_o = df_in.copy()
        df_o["q_value"] = q
        df_o["significant_q<0.05"] = (df_o["q_value"] < 0.05).fillna(False)
        return df_o

    if fdr_mode == "global":
        out = _apply_bh(out)
    elif fdr_mode == "by_method":
        # Aplica BH separadamente por “família” de teste (boa prática)
        out = out.groupby("method", group_keys=False).apply(_apply_bh).reset_index(drop=True)
    elif fdr_mode == "by_pair":
        # Útil se você gera vários testes por par e quer corrigir dentro do par
        out["_pair"] = out["x"] + " ~ " + out["y"]
        out = out.groupby("_pair", group_keys=False).apply(_apply_bh).reset_index(drop=True).drop(columns="_pair")
    else:
        warnings.warn("fdr_mode desconhecido; usando 'global'.")
        out = _apply_bh(out)

    # 6) ordena para facilitar leitura (significativos primeiro)
    out = out.sort_values(
        by=["significant_q<0.05", "q_value", "p_value"],
        ascending=[False, True, True],
        na_position="last"
    ).reset_index(drop=True)

    return out

In [7]:
# ---------------------------------------------------------------------
# [BLOCO 5] Rótulos intuitivos para o tamanho do efeito (heurísticos)
# ---------------------------------------------------------------------
def add_effect_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Acrescenta a coluna 'effect_strength' com rótulos amigáveis:
      - Correlações (|r|): 0.1 fraco, 0.3 médio, 0.5 forte
      - Cramér V: 0.1 fraco, 0.3 médio, 0.5 forte
      - Eta²: 0.01 pequeno, 0.06 médio, 0.14 grande (Cohen)
      - dCor: 0.1 fraco, 0.3 médio, 0.5 forte (heurístico)
      - Spline vs Linear: usa R² do spline (≈ 0.02/0.13/0.26 → muito pequeno/pequeno/médio/grande)
    """
    def _label(m, e):
        if pd.isna(e):
            return "n/a"
        e = abs(float(e))
        if m in ("pearson", "spearman", "kendall_tau"):
            return "muito fraco" if e < 0.1 else "fraco" if e < 0.3 else "médio" if e < 0.5 else "forte"
        if m == "cramers_v":
            return "muito fraco" if e < 0.1 else "fraco" if e < 0.3 else "médio" if e < 0.5 else "forte"
        if m == "eta_squared":
            return "muito pequeno" if e < 0.01 else "pequeno" if e < 0.06 else "médio" if e < 0.14 else "grande"
        if m == "distance_corr":
            return "muito fraco" if e < 0.1 else "fraco" if e < 0.3 else "médio" if e < 0.5 else "forte"
        if m.startswith("spline_vs_linear"):
            return "muito pequeno" if e < 0.02 else "pequeno" if e < 0.13 else "médio" if e < 0.26 else "grande"
        return "n/a"

    dfx = df.copy()
    dfx["effect_strength"] = [_label(m, e) for m, e in zip(dfx["method"], dfx["effect"])]
    return dfx


# Executando

In [10]:
rng = np.random.default_rng(42)
n = 400
x1 = rng.normal(size=n)
y1 = 0.8*x1 + rng.normal(scale=0.4, size=n)            # forte linear
x2 = rng.uniform(-2, 2, size=n)
y2 = 0.5*(x2**2) + rng.normal(scale=0.3, size=n)       # não-linear
cat = pd.Categorical(rng.choice(["A", "B", "C"], size=n, p=[0.4, 0.4, 0.2]))
num = rng.normal(loc=np.select([cat=="A", cat=="B", cat=="C"], [0.0, 0.5, 1.0]), scale=1.0)
binv = pd.Categorical(rng.choice([0, 1], size=n))
df_demo = pd.DataFrame({"x1":x1,"y1":y1,"x2":x2,"y2":y2,"num":num,"cat":cat,"binv":binv})

In [11]:
df_demo.head()

,x1,y1,x2,y2,num,cat,binv
0,0.304717,0.171929,0.443081,0.678298,-0.360243,C,1
1,-1.039984,-0.753277,0.214163,0.611078,0.890725,B,1
2,0.750451,0.928572,-0.415334,-0.282149,1.045323,B,0
3,0.940565,0.594955,0.710483,-0.025567,1.231660,C,1
4,-1.951035,-1.352361,0.903079,0.853251,0.965170,A,0


In [14]:
res = scan_associations(
    df_demo,
    permutations=500,          # ativa permutação para dCor e eta²
    fdr_mode="by_method",      # corrige por família de testes (boa prática)
    spline_test="wald_robust", # teste de não-linearidade robusto (HC3)
    progress=True
)
res = add_effect_labels(res)

Pares:   0%|          | 0/21 [00:00<?, ?it/s]

C:\Users\gcabr\AppData\Local\Temp\ipykernel_10932\3373833484.py:20: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(s) and getattr(s.dtype, "ordered", False):
C:\Users\gcabr\AppData\Local\Temp\ipykernel_10932\3373833484.py:30: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(s) or pd.api.types.is_object_dtype(s):
C:\Users\gcabr\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:1889: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
C:\Users\gcabr\AppData\Local\Programs\Python\Py

In [19]:
res.head()

,x,y,type_pair,method,effect,p_value,n,notes,q_value,significant_q<0.05,effect_strength
0,x2,y2,numeric-numeric,spline_vs_linear_waldHC3,0.750968,0.000000e+00,400,,0.000000e+00,True,grande
1,x1,y1,numeric-numeric,spline_vs_linear_waldHC3,0.771495,1.323978e-282,400,,6.619888e-282,True,grande
2,x1,y1,numeric-numeric,pearson,0.877982,2.093022e-129,400,,2.093022e-128,True,forte
3,x1,y1,numeric-numeric,spearman,0.863350,2.756515e-120,400,,2.756515e-119,True,forte
4,x1,y1,numeric-numeric,kendall_tau,0.681153,5.084230e-92,400,,5.084230e-91,True,forte
